<h3> Path setup and imports </h3>

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PYTHONPATH:", PROJECT_ROOT)


PYTHONPATH: /Users/aidos/ML Projects Personal/Comp_BioChem_Project


In [2]:
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from src.baselines.features import FeatureConfig, build_features_for_chain, load_labels, load_graph
from src.models.gnn import GNNConfig, InterfaceGNN

ROOT = PROJECT_ROOT
META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


Device: mps


<h3> Load split</h3>

In [3]:
with open(META / "data_splits.json", "r") as f:
    splits = json.load(f)

len(splits["train"]), len(splits["val"]), len(splits["test"])


(180, 35, 35)

<h3> Dataset class (returns one chain-graph at a time)

In [6]:
class ChainGraphExample:
    def __init__(self, x, edge_index, edge_dist, y, meta):
        self.x = x
        self.edge_index = edge_index
        self.edge_dist = edge_dist
        self.y = y
        self.meta = meta

class PPICChainDataset(Dataset):
    def __init__(self, split_list, split_name: str, t_angstrom=5):
        self.items = []
        self.cfg = FeatureConfig(include_flags=True, include_position=True)

        for ex in split_list:
            ex_dir = PROCESSED / f'{ex["pdb_id"]}_{ex["chainA"]}_{ex["chainB"]}'
            for chain in ["A", "B"]:
                X, feat_names = build_features_for_chain(ex_dir, chain, cfg=self.cfg)
                y = load_labels(ex_dir, chain, t_angstrom=t_angstrom)

                edge_index, edge_dist = load_graph(ex_dir, chain)

                # convert to torch
                x_t = torch.tensor(X, dtype=torch.float32)
                y_t = torch.tensor(y, dtype=torch.float32)

                ei_t = torch.tensor(edge_index, dtype=torch.long)
                ed_t = torch.tensor(edge_dist, dtype=torch.float32)

                meta = {"example": ex_dir.name, "chain": chain, "split": split_name}
                self.items.append(ChainGraphExample(x_t, ei_t, ed_t, y_t, meta))

        self.feat_names = feat_names

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

def collate_one(batch):
    # We train one graph at a time for simplicity and clarity.
    assert len(batch) == 1
    return batch[0]


<h3> Building Loaders</h3>

In [7]:
train_ds = PPICChainDataset(splits["train"], "train", t_angstrom=5)
val_ds   = PPICChainDataset(splits["val"],   "val",   t_angstrom=5)
test_ds  = PPICChainDataset(splits["test"],  "test",  t_angstrom=5)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_one)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=collate_one)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, collate_fn=collate_one)

len(train_ds), len(val_ds), len(test_ds), train_ds.feat_names[:5]


(360, 70, 70, ['aa_A', 'aa_R', 'aa_N', 'aa_D', 'aa_C'])

<h3> compute training interface fraction</h3>

In [8]:
import numpy as np
import torch

def compute_train_interface_fraction(train_ds):
    total_pos = 0
    total_all = 0
    for item in train_ds.items:
        total_pos += int(item.y.sum().item())
        total_all += int(item.y.numel())
    return total_pos / max(1, total_all)

train_iface_frac = compute_train_interface_fraction(train_ds)
train_iface_frac


0.11651469098277609

<h3> Compute pos_weight from train split (weighted BCE)</h3>

In [9]:
# pos_weight = (#neg / #pos) computed over all train residues
pos = 0
neg = 0
for item in train_ds.items:
    pos += int(item.y.sum().item())
    neg += int((item.y.numel() - item.y.sum()).item())

pos_weight = torch.tensor([neg / max(1, pos)], dtype=torch.float32, device=device)
pos, neg, pos_weight


(9430, 71504, tensor([7.5826], device='mps:0'))

<h3> Initializing Model + Optimizer

In [10]:
in_dim = train_ds.items[0].x.shape[1]
cfg = GNNConfig(in_dim=in_dim, hidden_dim=64, num_layers=3, num_rbf=12, rbf_dmin=2.0, rbf_dmax=10.0, dropout=0.10)

model = InterfaceGNN(cfg).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)


<h3> Evaluation metrics helpers (PR-AUC + Precision@K)

In [11]:
from sklearn.metrics import average_precision_score
import numpy as np
import torch

def contact_degree(n_nodes: int, edge_index: torch.Tensor) -> torch.Tensor:
    # edge_index: (2, E), src = receiver node
    src = edge_index[0]
    deg = torch.zeros(n_nodes, device=edge_index.device, dtype=torch.float32)
    deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float32))
    return deg

def pick_k_dynamic(n_nodes: int, frac: float, k_min: int = 5, k_max: int = 60) -> int:
    k = int(round(frac * n_nodes))
    k = max(k_min, min(k_max, k))
    k = min(k, n_nodes)
    return k

def topk_with_mask(score: np.ndarray, k: int, mask: np.ndarray | None = None) -> np.ndarray:
    # mask: True = allowed candidate
    if mask is not None:
        masked_score = score.copy()
        masked_score[~mask] = -1e9
    else:
        masked_score = score

    k = max(1, min(int(k), len(score)))
    idx = np.argsort(-masked_score)[:k]
    pred = np.zeros(len(score), dtype=np.int8)
    pred[idx] = 1
    return pred

def compute_metrics_from_scores(y_true: np.ndarray, score: np.ndarray, k: int, mask: np.ndarray | None = None):
    pred = topk_with_mask(score, k, mask)

    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()

    prec = tp / max(1, (tp + fp))
    rec  = tp / max(1, (tp + fn))
    f1   = 0.0 if (prec + rec) == 0 else (2 * prec * rec / (prec + rec))
    return float(prec), float(f1)

def eval_loader(
    loader,
    k_list=(10, 20, 30),
    surface_quantile: float | None = None,
    dynamic_k: bool = False,
    dynamic_frac: float = 0.12,
):
    """
    surface_quantile:
      None -> no mask
      e.g. 0.40 -> allow only lowest-degree 40% residues (surface-like proxy)

    dynamic_k:
      False -> evaluate fixed K in k_list
      True  -> also evaluate K_dynamic = round(dynamic_frac * N) per chain
    """
    model.eval()
    praucs = []
    p_at = {k: [] for k in k_list}
    f1_at = {k: [] for k in k_list}

    dyn_p = []
    dyn_f1 = []
    dyn_k_used = []

    with torch.no_grad():
        for item in loader:
            x = item.x.to(device)
            ei = item.edge_index.to(device)
            ed = item.edge_dist.to(device)
            y = item.y.to(device)

            logits = model(x, ei, ed)
            score = torch.sigmoid(logits).detach().cpu().numpy()
            y_np  = y.detach().cpu().numpy().astype(np.int8)

            praucs.append(float(average_precision_score(y_np, score)))

            # build surface mask if requested (based on low contact degree)
            mask = None
            if surface_quantile is not None:
                deg = contact_degree(len(y_np), ei).detach().cpu().numpy()
                thr = np.quantile(deg, surface_quantile)
                mask = (deg <= thr)

            # fixed K metrics
            for k in k_list:
                prec, f1 = compute_metrics_from_scores(y_np, score, k, mask=mask)
                p_at[k].append(prec)
                f1_at[k].append(f1)

            # dynamic K metrics
            if dynamic_k:
                k_dyn = pick_k_dynamic(len(y_np), dynamic_frac)
                prec_dyn, f1_dyn = compute_metrics_from_scores(y_np, score, k_dyn, mask=mask)
                dyn_p.append(prec_dyn)
                dyn_f1.append(f1_dyn)
                dyn_k_used.append(k_dyn)

    out = {"prauc": float(np.mean(praucs))}
    for k in k_list:
        out[f"p@{k}"] = float(np.mean(p_at[k]))
        out[f"f1@{k}"] = float(np.mean(f1_at[k]))

    if dynamic_k:
        out["p@Kdyn"] = float(np.mean(dyn_p))
        out["f1@Kdyn"] = float(np.mean(dyn_f1))
        out["Kdyn_mean"] = float(np.mean(dyn_k_used))

    return out


<h3> Training Loop (early stopping on val PR-AUC)</h3>

In [12]:
best_val = -1.0
best_state = None

for epoch in range(1, 21):
    model.train()
    losses = []

    for item in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        x = item.x.to(device)
        ei = item.edge_index.to(device)
        ed = item.edge_dist.to(device)
        y = item.y.to(device)

        opt.zero_grad()
        logits = model(x, ei, ed)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step()

        losses.append(float(loss.item()))

    val_metrics = eval_loader(val_loader)
    train_loss = float(np.mean(losses))

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_prauc={val_metrics['prauc']:.4f} | "
          f"val_p@10={val_metrics['p@10']:.3f} val_p@20={val_metrics['p@20']:.3f} val_p@30={val_metrics['p@30']:.3f}")

    if val_metrics["prauc"] > best_val:
        best_val = val_metrics["prauc"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# restore best
model.load_state_dict(best_state)

print("Best val PR-AUC:", best_val)


epoch 1:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 01 | train_loss=1.3772 | val_prauc=0.3783 | val_p@10=0.450 val_p@20=0.434 val_p@30=0.415


epoch 2:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 02 | train_loss=1.3227 | val_prauc=0.3751 | val_p@10=0.420 val_p@20=0.429 val_p@30=0.403


epoch 3:   0%|          | 0/360 [00:00<?, ?it/s]

...

Epoch 18 | train_loss=1.0542 | val_prauc=0.4053 | val_p@10=0.456 val_p@20=0.437 val_p@30=0.419


epoch 19:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 19 | train_loss=1.0413 | val_prauc=0.3969 | val_p@10=0.443 val_p@20=0.440 val_p@30=0.427


epoch 20:   0%|          | 0/360 [00:00<?, ?it/s]

Epoch 20 | train_loss=1.0344 | val_prauc=0.3950 | val_p@10=0.411 val_p@20=0.399 val_p@30=0.396
Best val PR-AUC: 0.42290404032962037


<h3> Final evaluation on test

In [13]:
val_final = eval_loader(val_loader)
test_final = eval_loader(test_loader)

val_final, test_final


({'prauc': 0.42290404032962037,
  'p@10': 0.5014285714285713,
  'f1@10': 0.12633813466300348,
  'p@20': 0.48000000000000015,
  'f1@20': 0.215579120884956,
  'p@30': 0.4671428571428571,
  'f1@30': 0.28382371737006534},
 {'prauc': 0.363423963873543,
  'p@10': 0.41714285714285715,
  'f1@10': 0.18754627648223451,
  'p@20': 0.3771428571428571,
  'f1@20': 0.25972264431210995,
  'p@30': 0.3523809523809524,
  'f1@30': 0.29375546279430637})

In [14]:
# Baseline evaluation (no tweaks)
val_base  = eval_loader(val_loader,  k_list=(10,20,30), surface_quantile=None, dynamic_k=False)
test_base = eval_loader(test_loader, k_list=(10,20,30), surface_quantile=None, dynamic_k=False)

# (1) Surface mask only (use same quantile as baseline heuristic: 0.40)
val_surf  = eval_loader(val_loader,  k_list=(10,20,30), surface_quantile=0.40, dynamic_k=False)
test_surf = eval_loader(test_loader, k_list=(10,20,30), surface_quantile=0.40, dynamic_k=False)

# (2) Dynamic K only (use train-derived interface fraction)
val_dyn   = eval_loader(val_loader,  k_list=(10,20,30), surface_quantile=None, dynamic_k=True, dynamic_frac=train_iface_frac)
test_dyn  = eval_loader(test_loader, k_list=(10,20,30), surface_quantile=None, dynamic_k=True, dynamic_frac=train_iface_frac)

# (1)+(2) Combined: surface mask + dynamic K
val_both  = eval_loader(val_loader,  k_list=(10,20,30), surface_quantile=0.40, dynamic_k=True, dynamic_frac=train_iface_frac)
test_both = eval_loader(test_loader, k_list=(10,20,30), surface_quantile=0.40, dynamic_k=True, dynamic_frac=train_iface_frac)

val_base, test_base, val_surf, test_surf, val_dyn, test_dyn, val_both, test_both


({'prauc': 0.42290404032962037,
  'p@10': 0.5014285714285713,
  'f1@10': 0.12633813466300348,
  'p@20': 0.48000000000000015,
  'f1@20': 0.215579120884956,
  'p@30': 0.4671428571428571,
  'f1@30': 0.28382371737006534},
 {'prauc': 0.36342127392832435,
  'p@10': 0.41714285714285715,
  'f1@10': 0.18754627648223451,
  'p@20': 0.3771428571428571,
  'f1@20': 0.25972264431210995,
  'p@30': 0.3523809523809524,
  'f1@30': 0.29375546279430637},
 {'prauc': 0.42290404032962037,
  'p@10': 0.49857142857142844,
  'f1@10': 0.1275023276129107,
  'p@20': 0.4749999999999999,
  'f1@20': 0.21618213493992103,
  'p@30': 0.44333333333333336,
  'f1@30': 0.2680512089368275},
 {'prauc': 0.36342127392832435,
  'p@10': 0.40285714285714286,
  'f1@10': 0.18256213015004064,
  'p@20': 0.35428571428571426,
  'f1@20': 0.23729024287884398,
  'p@30': 0.32238095238095243,
  'f1@30': 0.260264125344194},
 {'prauc': 0.42290404032962037,
  'p@10': 0.5014285714285713,
  'f1@10': 0.12633813466300348,
  'p@20': 0.48000000000000015

In [15]:
torch.save(model.state_dict(), META / "gnn_best.pt")
